# TCR / RQD Automatic Calculator - Demo

Complete pipeline demonstration:
1. Run segmentation (detect horizontal trays)
2. Piece detection (detect core pieces within runs)
3. TCR & RQD calculation

Uses production modules from `utils/`

In [ ]:
import sys
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import json

# Add parent directory to import utils
sys.path.insert(0, os.path.abspath('..'))

from utils.run_segmentation import extract_runs
from utils.piece_detection import detect_pieces
from utils.metrics import compute_lengths, compute_metrics

## 1. Load Image

In [ ]:
# Load a sample image
image_path = '../data/1.png'
image = cv2.imread(image_path)

if image is None:
    raise ValueError(f"Could not load image from {image_path}")

print(f"Image shape: {image.shape}")

# Display original
plt.figure(figsize=(15, 8))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title('Original Core Box Image')
plt.axis('off')
plt.show()

## 2. Run Segmentation

Detect horizontal runs using projection-based method:
- Grayscale + Gaussian blur
- Row-wise mean projection
- Otsu threshold on 1D signal
- Split at dark valleys (metal separators)

In [ ]:
# Extract runs
runs = extract_runs(image)
print(f"Detected {len(runs)} runs")

# Display each run
fig, axes = plt.subplots(len(runs), 1, figsize=(15, 3 * len(runs)))
if len(runs) == 1:
    axes = [axes]

for i, run in enumerate(runs):
    axes[i].imshow(cv2.cvtColor(run, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f'Run {i+1} - Shape: {run.shape}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 3. Piece Detection

Detect individual core pieces within each run:
- Crop margins (labels, tray edges, depth markers)
- Otsu binary threshold → rock mask
- Canny edge detection → crack signal
- Column-wise projection: rock - edge penalty
- Otsu threshold on 1D score → piece boundaries

In [ ]:
# Process each run
results = []

for i, run in enumerate(runs):
    boxes = detect_pieces(run)
    lengths = compute_lengths(boxes, run, run_length_cm=150.0)
    metrics = compute_metrics(lengths, run_length_cm=150.0)
    
    results.append({
        'run': i + 1,
        'boxes': boxes,
        'lengths': lengths,
        'metrics': metrics
    })
    
    print(f"\nRun {i+1}:")
    print(f"  Pieces: {metrics['n_pieces']}")
    print(f"  TCR: {metrics['tcr']:.2f}%")
    print(f"  RQD: {metrics['rqd']:.2f}%")
    print(f"  Recovered: {metrics['total_length']:.1f} cm")

## 4. Visualize Results

In [ ]:
# Draw bounding boxes on runs
fig, axes = plt.subplots(len(runs), 1, figsize=(16, 3 * len(runs)))
if len(runs) == 1:
    axes = [axes]

for i, (run, result) in enumerate(zip(runs, results)):
    # Draw boxes
    canvas = run.copy()
    for j, (x, y, w, h) in enumerate(result['boxes']):
        cv2.rectangle(canvas, (x, y), (x+w, y+h), (0, 255, 0), 2)
        if j < len(result['lengths']):
            label = f"{result['lengths'][j]:.1f}cm"
            cv2.putText(canvas, label, (x, max(y-5, 15)),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    
    axes[i].imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    m = result['metrics']
    axes[i].set_title(
        f"Run {i+1} — TCR: {m['tcr']:.1f}% | RQD: {m['rqd']:.1f}% | "
        f"{m['n_pieces']} pieces"
    )
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 5. Summary Statistics

In [ ]:
# Calculate summary
tcr_vals = [r['metrics']['tcr'] for r in results]
rqd_vals = [r['metrics']['rqd'] for r in results]
total_pieces = sum(r['metrics']['n_pieces'] for r in results)

print("=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"Total Runs: {len(results)}")
print(f"Total Pieces: {total_pieces}")
print(f"Average TCR: {np.mean(tcr_vals):.2f}%")
print(f"Average RQD: {np.mean(rqd_vals):.2f}%")
print(f"TCR Range: {min(tcr_vals):.1f}% - {max(tcr_vals):.1f}%")
print(f"RQD Range: {min(rqd_vals):.1f}% - {max(rqd_vals):.1f}%")

## 6. Export Results

In [ ]:
# Export to JSON
export = {
    'source': os.path.basename(image_path),
    'run_length_cm': 150.0,
    'n_runs': len(results),
    'summary': {
        'avg_tcr': round(float(np.mean(tcr_vals)), 2),
        'avg_rqd': round(float(np.mean(rqd_vals)), 2),
        'total_pieces': total_pieces
    },
    'runs': [
        {
            'run': r['run'],
            'lengths_cm': [round(l, 2) for l in r['lengths']],
            'metrics': r['metrics']
        }
        for r in results
    ]
}

output_path = '../outputs/demo_results.json'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, 'w') as f:
    json.dump(export, f, indent=2)

print(f"Results saved to: {output_path}")